In [28]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline, PruneGraph
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
# from circuit_tracer.subgraph.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [29]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [30]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [31]:
explainer = shap.Explainer(model, tokenizer)

In [32]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [33]:
shap_values = explainer(prompts)

In [51]:
shap.plots.text(shap_values)

In [52]:
print(shap_values)

.values =
array([[[-4.22446732e-06],
        [ 6.54286429e+00],
        [ 1.68508452e+00],
        [ 1.20230146e+00],
        [ 6.04736818e+00],
        [ 6.77137808e-02],
        [ 4.40673760e+00],
        [ 4.12085123e-01],
        [ 4.40843961e-01]]])

.base_values =
array([[-21.73993724]])

.data =
(array(['', 'Cat', ' is', ' to', ' kitten', ' as', ' dog', ' is', ' to'],
      dtype=object),)


In [34]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [4]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="austin_plt",
    source_set = 'gemmascope-transcoder-16k',
    out_path="temp_graph_files/austin_plt.json",
    node_threshold = 0.95,
    edge_threshold = 0.98,
)

requesting graph for slug='austin_plt' prompt='Fact: The capital of the state containing Dallas is...'


downloading graph json from https://neuronpedia-attrib.s3.us-east-1.amazonaws.com/user-graphs/anonymous/austin_plt.json
saved austin_plt -> temp_graph_files/austin_plt.json (nodes=5387)


## ASG Pipeline

### Prune

In [ ]:
json_path = "temp_graph_files/austin_plt.json"
# json_path = "temp_graph_files/future-qwen.json"
# source_set = 'gemmascope-transcoder-16k' #'clt-hp' # gemmascope-transcoder-16k
# token_weights = [0.00198786, 0.03153391, 0.00083086, 0.01473883, 0.22338926, 0.00649094,
#  0.00222269, 0.01996207, 0.0052309, 0.67869559, 0.01491708]
# token_weights = [0, 1/3, 0, 0, 1/3, 0, 1/3, 0, 0]
# token_weights = [0, 0, 1/4, 0, 0, 1/4, 0, 1/4, 1/4, 0, 0]
# token_weights = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1/3,0,0,1/3,0,1/3,0,0,0,0,0,0,0,0,0,0,0]
prune_graph = prune_graph_pipeline(
    json_path=json_path,
    logit_weights='target',
    token_weights=token_weights,
    node_influence_threshold=0.6,
    edge_influence_threshold=0.9,
    node_relevance_threshold=0.8,
    edge_relevance_threshold=0.9,
    keep_all_tokens_and_logits=False,
    filter_act_density=True,
)

In [85]:
print(len(prune_graph.kept_ids))

30


In [88]:
for i, node_id in enumerate(prune_graph.kept_ids):
    print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_influence[i].item(), prune_graph.node_relevance[i].item())

1_11076_4  words related to veterinary medicine, especially diseases in farm animals 0.537501871585846 0.7895568013191223
4_11720_4  the word "cat" including related terms 0.5417401790618896 0.7658317685127258
7_11190_6 words related to dogs and associated actions in a fairly sexual context. 0.49491170048713684 0.7883498668670654
8_1093_6  the virus Porcine Reproductive and Respiratory Syndrome (PRRSV) and related immunology terms. 0.5816071033477783 0.7807872891426086
14_13415_4  words related to animals, particularly dogs, including their emotions and health 0.5662703514099121 0.6994884610176086
14_13415_6  words related to animals, particularly dogs, including their emotions and health 0.48927563428878784 0.7149860858917236
16_3708_6 dog 0.5673236846923828 0.7257762551307678
16_3708_8 dog 0.56191486120224 0.7207479476928711
18_595_6  words related to dogs and dog parks 0.5194127559661865 0.6020478010177612
18_1686_6 references to dogs and children together 0.5153667330741882 0.71790

In [1]:
save_prune_snapshot('subgraph/puppy_plt_shap_06_08.pt')

NameError: name 'save_prune_snapshot' is not defined

In [ ]:
prune_graph = PruneGraph.load_prune_snapshot('subgraph/puppy_clt_shap_07_085.pt')

In [9]:
prune_graph.metadata

{'slug': 'factthecapitalof-1771775020852',
 'scan': 'gemma-2-2b',
 'transcoder_list': [],
 'prompt_tokens': ['<bos>',
  'Fact',
  ':',
  ' The',
  ' capital',
  ' of',
  ' the',
  ' state',
  ' containing',
  ' Dallas',
  ' is'],
 'prompt': '<bos>Fact: The capital of the state containing Dallas is',
 'node_threshold': 1,
 'schema_version': 1,
 'info': {'creator_name': 'tu7pham7',
  'creator_url': 'https://neuronpedia.org',
  'source_urls': ['https://neuronpedia.org/gemma-2-2b/clt-hp',
   'https://huggingface.co/mntss/clt-gemma-2-2b-2.5M'],
  'transcoder_set': 'mntss/clt-gemma-2-2b-2.5M',
  'generator': {'name': 'circuit-tracer by Hanna & Piotrowski',
   'version': '0.3.1 | e09b5f3',
   'url': 'https://github.com/safety-research/circuit-tracer'},
  'create_time_ms': 1771775033362,
  'neuronpedia_link': 'https://www.neuronpedia.org/gemma-2-2b/graph?slug=factthecapitalof-1771775020852&pruningThreshold=1&densityThreshold=1',
  'neuronpedia_source_set': 'clt-hp'},
 'generation_settings': {'

### Classify features

Heuristic classify alg is terrible, needs improvement or just use LLM

In [ ]:
# feature_types = classify_features(kept_ids, attr, metadata)
# print(feature_types)

IndexError: list index out of range

In [ ]:
# node_ids = [node_id for node_id in kept_ids if attr[node_id].get("feature_type", "") == "cross layer transcoder"]
# # remove 21_61721_10, 22_74186_10
# node_ids = [node_id for node_id in node_ids if node_id not in ["21_61721_10", "22_74186_10"]]
# print(node_ids)

['16_89970_9', '18_45367_10', '18_56027_10', '19_8271_10', '19_62362_10', '20_44686_9', '20_14775_10', '20_44686_10', '20_74108_10', '21_16875_10', '21_84338_10', '22_11998_10', '22_26199_10', '22_32893_10', '22_43081_10', '22_81571_10', '23_29602_10', '23_31673_10', '23_59746_10', '24_79427_10']


In [ ]:
# labels = {node_id: attr[node_id].get("clerp", "") for node_id in node_ids }
# print(labels)

{'16_89970_9': 'Texas', '18_45367_10': 'US cities', '18_56027_10': 'Dallas', '19_8271_10': 'Foreign words', '19_62362_10': 'city names', '20_44686_9': 'Texas locations/legal contexts', '20_14775_10': 'XML code', '20_44686_10': 'Texas locations/legal contexts', '20_74108_10': 'capital', '21_16875_10': 'cities', '21_84338_10': 'Cities and locations', '22_11998_10': 'Texas and Dallas', '22_26199_10': 'Texas', '22_32893_10': 'Texas legal documents', '22_43081_10': 'technical contexts', '22_81571_10': 'Texas locations and organizations', '23_29602_10': 'City or region names', '23_31673_10': 'Court appeals at specific location', '23_59746_10': 'Technical/Scientific writing', '24_79427_10': ' Kara'}


In [ ]:
# feature_types = {
#     node_id: "Abstract" for node_id in node_ids
# }
# feature_types['1_89326_9'] = "Input"
# print(feature_types)

{'16_89970_9': 'Abstract', '18_45367_10': 'Abstract', '18_56027_10': 'Abstract', '19_8271_10': 'Abstract', '19_62362_10': 'Abstract', '20_44686_9': 'Abstract', '20_14775_10': 'Abstract', '20_44686_10': 'Abstract', '20_74108_10': 'Abstract', '21_16875_10': 'Abstract', '21_84338_10': 'Abstract', '22_11998_10': 'Abstract', '22_26199_10': 'Abstract', '22_32893_10': 'Abstract', '22_43081_10': 'Abstract', '22_81571_10': 'Abstract', '23_29602_10': 'Abstract', '23_31673_10': 'Abstract', '23_59746_10': 'Abstract', '24_79427_10': 'Abstract', '1_89326_9': 'Input'}


In [ ]:
# supernodes = grouping_pipeline(
#     node_ids = node_ids,
#     labels = labels,
#     feature_types = feature_types,
#     prompt = prompts[0],
#     model_name = 'gemini-2.5-flash',
# )
# print(supernodes)

[['Texas State Context', '16_89970_9', '22_11998_10', '22_26199_10'], ['General Geographic Entities', '18_45367_10', '19_62362_10', '21_16875_10', '21_84338_10', '23_29602_10'], ['Texas Legal/Specific Contexts', '20_44686_9', '20_44686_10', '22_32893_10', '22_81571_10', '23_31673_10'], ['Miscellaneous/Irrelevant', '19_8271_10', '20_14775_10', '22_43081_10', '23_59746_10', '24_79427_10']]


In [42]:
from circuit_tracer.subgraph.prune import prune_graph_pipeline
from circuit_tracer.subgraph.cluster import cluster_graph, cluster_graph_with_labels
from circuit_tracer.subgraph.auto_grouping import find_best_k

# token_weights = [0,0,0,0,1/3,0,0,1/3,0,1/3,0]
# 1) Build prune_graph as usual
prune_graph = prune_graph_pipeline(
    json_path="temp_graph_files/austin_plt.json",
    token_weights = token_weights,
    logit_weights="target",
    node_influence_threshold=0.7,
    edge_influence_threshold=0.98,
    node_relevance_threshold=0.7,
    edge_relevance_threshold=0.98,
    keep_all_tokens_and_logits=False,
    filter_act_density=True,
)


Failed node=4_13154_9 modelId=gemma-2-2b layer=4- status=500 body=
Failed node=14_2268_9 modelId=gemma-2-2b layer=14- status=500 body=
Failed node=16_25_9 modelId=gemma-2-2b layer=16- status=500 body=
Failed node=16_12678_9 modelId=gemma-2-2b layer=16- status=500 body=
Failed node=16_15223_10 modelId=gemma-2-2b layer=16- status=500 body=
Failed node=18_187_9 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_187_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_1026_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_1437_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_3852_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_5495_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_6101_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_8959_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=18_16041_10 modelId=gemma-2-2b layer=18- status=500 body=
Failed node=19_7477_9 modelId=gemma-2-2b 

KeyboardInterrupt: 

In [ ]:
prune_graph.num_nodes, prune_graph.num_edges

(31, 196)

In [ ]:
from circuit_tracer.subgraph.prune import save_prune_graph, load_prune_graph
save_prune_graph(prune_graph, "subgraph/austin_plt_shap_07_07.pt")
# loaded_prune_graph = load_prune_graph("temp_graph_files/austin_clt_prune

In [38]:
# 2) Auto-select k
best_k, sweep = find_best_k(
    prune_graph,
    max_layer_span=4,
    # optional knobs:
    # k_min_override=3,
    # k_max_override=12,
    # max_sn=10,
    # weights={"w_intra": 0.3, "w_dag": 0.25, "w_flow": 0.25, "w_size": 0.2},
)

print("best_k:", best_k)
print("best score:", sweep[best_k]["total"] if best_k in sweep else None)

# 3) Run final clustering with best_k
supernodes = cluster_graph_with_labels(
    prune_graph,
    target_k=best_k,
    max_layer_span=4,
)

best_k: 2
best score: 0.860178854874986


In [39]:
supernodes

[['cluster_0', '0_32742_9', '1_89326_9'],
 ['cluster_1', '7_24743_9', '8_21080_9'],
 ['cluster_2', '16_89970_9', '16_37465_10', '17_98126_10'],
 ['cluster_4',
  '22_11998_10',
  '22_32893_10',
  '22_81571_10',
  '18_61980_10',
  '22_74186_10'],
 ['cluster_5', '18_3623_10', '20_14775_10', '19_54790_10', '21_84338_10'],
 ['cluster_6', '20_44686_9', '20_44686_10', '18_45367_10'],
 ['cluster_7', '22_43081_10', '23_31673_10']]

In [40]:
for supernode in supernodes:
    print(supernode[0])
    for node_id in supernode[1:]:
        print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_influence[prune_graph.kept_ids.index(node_id)].item(), prune_graph.node_relevance[prune_graph.kept_ids.index(node_id)].item())

cluster_0
0_32742_9 places 0.6424604654312134 0.6853705048561096
1_89326_9 Dallas 0.6522813439369202 0.6492207646369934
cluster_1
7_24743_9 Texas legal matters 0.6338451504707336 0.5306751132011414
8_21080_9 Texas legal contexts 0.6120493412017822 0.6154094338417053
cluster_2
16_89970_9 Texas 0.6929287910461426 0.6952129602432251
16_37465_10 locations and names 0.699770987033844 0.5970065593719482
17_98126_10 Government buildings/institutions 0.6986528038978577 0.6960087418556213
cluster_4
22_11998_10 Texas and Dallas 0.5273332595825195 0.5067331790924072
22_32893_10 Texas legal documents 0.5873730778694153 0.5807951092720032
22_81571_10 Texas locations and organizations 0.5839060544967651 0.5113580226898193
18_61980_10 Texas related 0.5764042735099792 0.6396741271018982
22_74186_10 general news snippets 0.6148593425750732 0.588416576385498
cluster_5
18_3623_10 capital cities 0.6403317451477051 0.582324743270874
20_14775_10 XML code 0.6360085010528564 0.6532292366027832
19_54790_10 Pla

In [7]:
import json
from typing import List
def extract_supernodes(flow_map_path: str) -> List[List[str]]:
    """
    Load a mapping JSON like flow_analysis_supernode_map.json and return
    a list of supernodes in the format:
      [ ["SN_LABEL", "node_a", "node_b", ...], ... ]
    """
    p = Path(flow_map_path)
    with p.open("r", encoding="utf-8") as f:
        mapping = json.load(f)

    supernodes = [[label, *nodes] for label, nodes in mapping.items()]
    return supernodes

### Visualize on the web

In [23]:
import json
from circuit_tracer.subgraph.api import save_subgraph

# Expected format for save_subgraph:
# supernodes = [
#   ["label_1", "node_id_a", "node_id_b"],
#   ["label_2", "node_id_c", "node_id_d", "node_id_e"],
# ]
# supernodes = extract_supernodes("flow_analysis_supernode_map.json")
# print(supernodes)


In [41]:
status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug="factthecapitalof-1771775020852",                       # parent graph slug
    displayName="austin-grouped-shap",     # subgraph name shown on Neuronpedia
    pinnedIds=prune_graph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.7,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

{'Content-Type': 'application/json', 'x-api-key': 'sk-np-Vi0OcQl8zNm9jC7nyGRYfwssvSakMyrPjV675uEWIuU0'}


status: 200
{'success': True, 'subgraphId': 'cmnybzrgv0002uz3gx86q78xb'}


# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality